# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark
import re
import os
import posixpath

# Parameters

As I'm using Databrick's "free edition" I can't read from any cloud storage provider (AWS, GCP, etc...). So, this project will read the files from Github repository:

In [0]:
SOURCE_PATH = "/Workspace/Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/datasets"

FORMAT_MAP = {
                 "csv": "csv",
                 "json": "json",
                 "parquet": "parquet",
                 "avro": "avro",
                 "orc": "orc",
                 "txt": "text",
                 "xml": "xml",
             }

# Ingesting Data

In [0]:
files = [
            posixpath.join(root, fn)
            for root, _, file_names in os.walk(SOURCE_PATH)
            for fn in file_names
            if not fn.startswith(".")
        ]


for file_path in files:
    file_name = posixpath.basename(file_path)

    # Derive format and table name from the file extension
    ext = file_name.rsplit(".", 1)[-1].lower() if "." in file_name else ""
    read_format = FORMAT_MAP.get(ext)
    if read_format is None:
        print(f"Skipping unsupported file type: {file_name}")
        continue

    table_name = file_name.rsplit(".", 1)[0].lower().replace("-", "_").replace(" ", "_")
    full_table = f"bronze.{table_name}"

    # Incremental guard: only ingest files whose bronze table does not exist yet
    if spark.catalog.tableExists(full_table):
        print(f"  ↷ {full_table} already exists — skipping (incremental mode)")
        continue

    print(f"\nProcessing {file_name}  →  {full_table}  (format: {read_format})")

    # --- Read the file ----------------------------------------------------------
    if read_format == "csv":
        # Sniff the field separator and encoding from the first line of the file
        with open(file_path, "rb") as fh:
            head = fh.read(4096).decode("ISO-8859-1", errors="replace")
        first_line = head.split("\n")[0]
        sep = ";" if first_line.count(";") > first_line.count(",") else ","
        encoding = "ISO-8859-1" if "\ufffd" in head else "UTF-8"
        df = (
                spark.read.option("header", "true")
                          .option("inferSchema", "false")
                          .option("escape", '"')
                          .option("sep", sep)
                          .option("encoding", encoding)
                          .csv(file_path)
             )
    elif read_format == "text":
        df = spark.read.text(file_path)
    elif read_format == "xml":
        df = (
                spark.read.option("rowTag", "row")
                          .format("xml")
                          .load(file_path)
             )
    else:
        reader = spark.read.format(read_format)
        if read_format == "json":
            reader = reader.option("multiLine", "true")
        df = reader.load(file_path)

    # --- Make column names Delta-safe (no spaces, ;(){} etc.) ------------------
    df = df.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df.columns])

    # --- Add bronze-layer metadata columns -------------------------------------
    df = df.select(
                    "*",
                    F.col("_metadata.file_name").alias("_source_file"),
                    F.col("_metadata.file_path").alias("_source_path"),
                    F.current_timestamp().alias("_ingestion_ts"),
                  )

    # --- Write to the bronze schema as a managed Delta table --------------------
    (
        df.write.format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(full_table)
    )

    row_count = spark.table(full_table).count()
    print(f"  ✓ Written {row_count} rows to {full_table}")

print("\n✅ All files processed.")
